
# CVM-A1: phân tích thời gian và bộ nhớ

Notebook có ba chế độ: `READ_EXISTING_P0` đọc bộ P0 đã đo trên Windows; `SMOKE_LINUX` chạy smoke test thật qua API TV3; `CUSTOM_SWEEP` chạy cấu hình đã chọn qua API TV3. Run All mặc định chỉ đọc P0.

Runtime thuật toán, thời gian đầu-cuối, RSS của tiến trình C++ và kích thước mẫu là các đại lượng khác nhau. Bộ P0 mô tả môi trường Windows trong sidecar. Số đo thực nghiệm hữu hạn giúp khảo sát xu hướng; chúng không tự chứng minh cận tiệm cận Big-O.


In [ ]:

from pathlib import Path
import importlib.util
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import uuid
from datetime import datetime, timezone

import pandas as pd
from IPython.display import Image, Markdown, display

def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc có pyproject.toml và src/.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXPORT_DIR = ROOT / "notebooks" / "exports"
P0_RAW = ROOT / "results" / "real" / "tv4_timespace_p0_20261113.csv"
P0_SIDECAR = P0_RAW.with_suffix(".metadata.json")

compiler_path = shutil.which("g++") or shutil.which("clang++")
compiler_version = "không tìm thấy GCC/Clang"
if compiler_path:
    compiler_result = subprocess.run(
        [compiler_path, "--version"], capture_output=True, text=True, check=False
    )
    compiler_version = (compiler_result.stdout or compiler_result.stderr).splitlines()[0]

display(Markdown("\n".join([
    f"**Kernel:** `{sys.executable}`",
    f"**Python:** `{platform.python_version()}`",
    f"**Hệ điều hành:** `{platform.platform()}`",
    f"**Trình biên dịch:** `{compiler_version}`",
])))



## Cấu hình chế độ chạy

Sửa `MODE` để chọn chế độ. `CUSTOM_SWEEP` mặc định chỉ xem trước số lượt; đặt `RUN_CUSTOM = True` để cho phép chạy benchmark. Các tham số custom không ảnh hưởng chế độ đọc P0.


In [ ]:

MODE = "READ_EXISTING_P0"  # READ_EXISTING_P0 | SMOKE_LINUX | CUSTOM_SWEEP
RUN_CUSTOM = False

# n và F0 là giá trị đơn; None yêu cầu TV3 suy ra giá trị phù hợp.
CUSTOM_M = [256, 1024]
CUSTOM_N = None
CUSTOM_F0 = None
CUSTOM_EPSILON = 0.05
CUSTOM_DELTA = 0.01
CUSTOM_WORKLOADS = ["uniform"]
CUSTOM_SEED_GENERATOR = 44000
CUSTOM_SEED_ALGORITHM = 84000
CUSTOM_TRIALS = 2

ALLOWED_MODES = {"READ_EXISTING_P0", "SMOKE_LINUX", "CUSTOM_SWEEP"}
if MODE not in ALLOWED_MODES:
    raise ValueError(f"MODE phải thuộc {sorted(ALLOWED_MODES)}, nhận được {MODE!r}.")

def validate_custom_config() -> None:
    u64_max = 2**64 - 1
    if not CUSTOM_M or any(
        isinstance(m, bool) or not isinstance(m, int) or not 1 <= m <= u64_max
        for m in CUSTOM_M
    ):
        raise ValueError("CUSTOM_M phải là danh sách số nguyên uint64 dương, không rỗng.")
    if not CUSTOM_WORKLOADS:
        raise ValueError("CUSTOM_WORKLOADS không được rỗng.")
    allowed_workloads = {"all_duplicate", "all_distinct", "uniform", "skewed", "zipf"}
    if any(workload not in allowed_workloads for workload in CUSTOM_WORKLOADS):
        raise ValueError("CUSTOM_WORKLOADS chứa workload không được hỗ trợ.")
    for name, value in (("epsilon", CUSTOM_EPSILON), ("delta", CUSTOM_DELTA)):
        if not isinstance(value, (int, float)) or not math.isfinite(value) or not 0 < value < 1:
            raise ValueError(f"{name} phải hữu hạn và thuộc (0, 1).")
    if isinstance(CUSTOM_TRIALS, bool) or not isinstance(CUSTOM_TRIALS, int) or CUSTOM_TRIALS < 1:
        raise ValueError("CUSTOM_TRIALS phải là số nguyên dương.")
    seeds = (CUSTOM_SEED_GENERATOR, CUSTOM_SEED_ALGORITHM)
    if any(isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed <= u64_max for seed in seeds):
        raise ValueError("Seed phải thuộc miền uint64.")
    if max(seeds) + CUSTOM_TRIALS - 1 > u64_max:
        raise ValueError("Lịch seed vượt miền uint64 khi xét số trial.")
    if CUSTOM_N is not None and (
        isinstance(CUSTOM_N, bool) or not isinstance(CUSTOM_N, int) or not 1 <= CUSTOM_N <= 2**64
    ):
        raise ValueError("CUSTOM_N phải thuộc [1, 2**64] hoặc là None.")
    if CUSTOM_F0 is not None and (
        isinstance(CUSTOM_F0, bool) or not isinstance(CUSTOM_F0, int) or not 1 <= CUSTOM_F0 <= u64_max
    ):
        raise ValueError("CUSTOM_F0 phải là uint64 dương hoặc là None.")
    for m in CUSTOM_M:
        if CUSTOM_F0 is not None and CUSTOM_F0 > m:
            raise ValueError(f"F0={CUSTOM_F0} vượt m={m}.")
        if CUSTOM_N is not None and CUSTOM_F0 is not None and CUSTOM_F0 > CUSTOM_N:
            raise ValueError(f"F0={CUSTOM_F0} vượt n={CUSTOM_N}.")
        for workload in CUSTOM_WORKLOADS:
            if workload == "all_distinct":
                if CUSTOM_N is not None and CUSTOM_N < m:
                    raise ValueError(f"all_distinct yêu cầu n >= m (m={m}, n={CUSTOM_N}).")
                if CUSTOM_F0 is not None and CUSTOM_F0 != m:
                    raise ValueError(f"all_distinct yêu cầu F0=m (m={m}, F0={CUSTOM_F0}).")
            if workload == "all_duplicate" and CUSTOM_F0 not in (None, 1):
                raise ValueError("all_duplicate yêu cầu F0=1.")

if MODE == "CUSTOM_SWEEP":
    validate_custom_config()



## Chạy chế độ và phân tích

Mỗi lần chạy tạo thư mục riêng dưới `notebooks/exports/`. Notebook xác thực RAW/sidecar trước khi phân tích. Nếu API Jupyter TV4 chưa có, notebook gọi các hàm tổng hợp và vẽ biểu đồ TV4 hiện có; notebook không chứa lại công thức phân tích.


In [ ]:

def new_run_dir(label: str) -> Path:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = EXPORT_DIR / f"{label}-{timestamp}-{uuid.uuid4().hex[:8]}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir

def require_native_ubuntu() -> list[str]:
    if platform.system() != "Linux":
        raise RuntimeError("Chế độ benchmark này chỉ chạy trên Linux native Ubuntu.")
    os_release = Path("/etc/os-release")
    release_fields = {}
    if os_release.is_file():
        for line in os_release.read_text(encoding="utf-8").splitlines():
            if "=" in line:
                key, value = line.split("=", 1)
                release_fields[key] = value.strip().strip('"')
    if release_fields.get("ID", "").lower() != "ubuntu":
        raise RuntimeError("Không xác nhận được Ubuntu qua /etc/os-release.")
    wsl_markers = ("WSL_DISTRO_NAME", "WSL_INTEROP")
    release_text = " ".join((Path("/proc/version").read_text(encoding="utf-8", errors="replace"),
                             Path("/proc/sys/kernel/osrelease").read_text(encoding="utf-8", errors="replace")))
    if any(os.environ.get(name) for name in wsl_markers) or "microsoft" in release_text.lower():
        raise RuntimeError("Phát hiện WSL; yêu cầu phép đo trên Linux native.")
    detector = shutil.which("systemd-detect-virt")
    if detector is None:
        raise RuntimeError("Thiếu systemd-detect-virt nên chưa xác nhận được môi trường native.")
    evidence = []
    for option in ("--vm", "--container"):
        result = subprocess.run([detector, option], capture_output=True, text=True, check=False)
        detected = (result.stdout or result.stderr).strip() or "none"
        evidence.append(f"{option}: {detected}")
        if result.returncode == 0 and detected.lower() != "none":
            raise RuntimeError(f"Phát hiện môi trường ảo hóa/container ({detected}); dừng benchmark.")
    return evidence

analysis_api_available = importlib.util.find_spec("scripts.jupyter_analysis") is not None
benchmark_api_available = importlib.util.find_spec("src.jupyter_adapter") is not None

def analyze_and_display(raw_path: Path, metadata_path: Path, output_dir: Path) -> dict:
    raw_path = raw_path.resolve()
    metadata_path = metadata_path.resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    if analysis_api_available:
        from scripts.jupyter_analysis import analyze_jupyter_results

        report = analyze_jupyter_results(raw_path, output_dir, metadata_path=metadata_path)
        raw_frame, summary_frame = report.raw, report.summary
        png_paths = tuple(report.png_paths)
        summary_path = report.summary_path
        classification = report.provenance.classification
        notes = report.notes
        metadata = report.provenance.metadata
    else:
        from scripts.analyze_results import summarize, write_summary
        from scripts.plot_results import plot_summary
        from scripts.validate_results import validate

        validation = validate(raw_path, metadata_path)
        if not validation.verified:
            raise ValueError("RAW/sidecar không qua validator:\n" + "\n".join(validation.errors))
        groups = summarize(validation.rows)
        raw_frame = pd.read_csv(raw_path)
        summary_frame = pd.DataFrame.from_records(groups)
        summary_path = output_dir / "summary.csv"
        write_summary(summary_path, groups)
        png_paths = tuple(plot_summary(groups, output_dir))
        classification = validation.classification
        notes = tuple(validation.warnings)
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

    display(Markdown("\n".join([
        f"**RAW:** `{raw_path}`",
        f"**Sidecar:** `{metadata_path}`",
        f"**Kiểm định:** `{classification}`",
    ])))
    display(raw_frame)
    display(summary_frame)
    if notes:
        display(Markdown("\n".join(f"- {note}" for note in notes)))
    for image_path in png_paths:
        display(Image(filename=str(image_path), width=900))
    environments = [batch.get("environment", {}) for batch in metadata.get("batches", [])]
    if environments:
        display(Markdown("**Môi trường trong sidecar:**\n" + "\n".join(
            f"- OS: {env.get('os', '—')}; Python: {env.get('python', '—')}; "
            f"CPU: {env.get('cpu', '—')}; compiler: {env.get('compiler', '—')}"
            for env in environments
        )))
    print(f"Bảng tổng hợp: {summary_path}")
    print(f"Thư mục hiện vật của lượt này: {output_dir}")
    return {"raw": raw_frame, "summary": summary_frame, "png_paths": png_paths,
            "summary_path": summary_path, "classification": classification}


In [ ]:

analysis_result = None
benchmark_result = None

if MODE == "READ_EXISTING_P0":
    if not P0_RAW.is_file() or not P0_SIDECAR.is_file():
        raise FileNotFoundError(f"Thiếu RAW hoặc sidecar P0: {P0_RAW}")
    run_dir = new_run_dir("read-existing-p0")
    analysis_result = analyze_and_display(P0_RAW, P0_SIDECAR, run_dir / "analysis")
    if len(analysis_result["raw"]) != 135:
        raise ValueError(f"P0 dự kiến có 135 bản ghi, nhận được {len(analysis_result['raw'])}.")
    if len(analysis_result["summary"]) != 15:
        raise ValueError(f"P0 dự kiến có 15 nhóm, nhận được {len(analysis_result['summary'])}.")
    if len(analysis_result["png_paths"]) != 3:
        raise ValueError(f"P0 dự kiến có ba hình, nhận được {len(analysis_result['png_paths'])}.")
    display(Markdown("**Nguồn:** Windows P0 đã lưu. Các bảng và hình vừa được tạo từ RAW/sidecar đã xác thực; đây không phải phép đo Linux."))

elif MODE in {"SMOKE_LINUX", "CUSTOM_SWEEP"}:
    if MODE == "CUSTOM_SWEEP" and not RUN_CUSTOM:
        configurations = len(CUSTOM_M) * len(CUSTOM_WORKLOADS)
        attempts = configurations * CUSTOM_TRIALS
        display(Markdown(f"**Chưa chạy benchmark.** Kế hoạch gồm {configurations} cấu hình × {CUSTOM_TRIALS} trial = {attempts} lượt. Đặt `RUN_CUSTOM = True` để chạy khi API TV3 đã có."))
    elif not benchmark_api_available:
        display(Markdown("**CHỜ API TV3:** `src.jupyter_adapter` chưa có trong checkout. Chưa chạy benchmark và chưa tạo RAW."))
    else:
        native_evidence = require_native_ubuntu()
        display(Markdown("**Môi trường chạy:** Linux Ubuntu; `systemd-detect-virt` không phát hiện máy ảo/container.\n\n" + "\n".join(f"- {item}" for item in native_evidence)))
        run_dir = new_run_dir(MODE.lower())
        from src.jupyter_adapter import run_jupyter_benchmark

        if MODE == "SMOKE_LINUX":
            benchmark_result = run_jupyter_benchmark(preset="smoke", output_dir=run_dir / "real")
        else:
            benchmark_result = run_jupyter_benchmark(
                m=CUSTOM_M,
                n=CUSTOM_N,
                f0=CUSTOM_F0,
                epsilon=CUSTOM_EPSILON,
                delta=CUSTOM_DELTA,
                workload=CUSTOM_WORKLOADS,
                trials=CUSTOM_TRIALS,
                seed_generator=CUSTOM_SEED_GENERATOR,
                seed_algorithm=CUSTOM_SEED_ALGORITHM,
                output_dir=run_dir / "real",
                tag="custom_sweep",
            )
        frame = benchmark_result.to_dataframe
        display(Markdown("\n".join([
            f"**RAW:** `{benchmark_result.csv_path}`",
            f"**Sidecar:** `{benchmark_result.sidecar_path}`",
            f"**Bản ghi:** {benchmark_result.records_count} · **Cấu hình:** {benchmark_result.configs_count}",
        ])))
        display(frame)
        display(frame["status"].value_counts(dropna=False).rename_axis("status").to_frame("count"))
        analysis_result = analyze_and_display(
            benchmark_result.csv_path,
            benchmark_result.sidecar_path,
            run_dir / "analysis",
        )

if MODE == "READ_EXISTING_P0":
    print(f"RAW nguồn: {P0_RAW}")
    print(f"Sidecar nguồn: {P0_SIDECAR}")
if MODE == "CUSTOM_SWEEP" and not RUN_CUSTOM:
    print("RUN_CUSTOM=False: notebook không mở tiến trình benchmark.")
if MODE in {"SMOKE_LINUX", "CUSTOM_SWEEP"} and not benchmark_api_available:
    print("API TV3 chưa có; không tạo dữ liệu giả.")



## Diễn giải và giới hạn

P0 là bộ đo Windows. Smoke/custom chỉ được báo cáo là phép đo Linux sau khi API TV3 chạy trên Ubuntu native và validator chấp nhận RAW cùng sidecar. Bảng và hình được tái tạo từ cùng RAW đã xác thực; thư mục từng lượt lưu riêng đường dẫn đầu ra.

Các phép đo cho thấy hành vi ở những cấu hình đã chạy. Kết luận tiệm cận cần lập luận riêng về chi phí cập nhật, các lần giảm mẫu và sức chứa mẫu. Smoke có quá ít mức `m` để dùng suy luận xu hướng.
